### Each milestone will have a description and some comments documenting my choices and tradeoffs / thought processes; additionally, completed pipeline for deployment will be in rocketrichard.py 
### NOTE: all of the thought processes can also be found in log.md
#### Below are some helper functions from rr.ipynb (initial testing environment)

In [22]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import ast

import matplotlib.pyplot as plt
import seaborn as sns

In [23]:
#clear csv of empty entries and format into a pandas csv
def clear_csv(csv_path):
    print(csv_path)
    df = pd.read_csv(csv_path, encoding='ascii')
    df = df.dropna()
    return df
rocket_richard = clear_csv("../data/formattedwebscraped/rocketrichard.csv")
rocket_richard

ranks = True   #global variable to be used later; at the top of the notebook for convenience

../data/formattedwebscraped/rocketrichard.csv


In [24]:
client = NHLClient()
def extractPlayerID(url):
    '''
    purpose:    parses a url and returns that player's id (used by the NHL API) 
    parameters: player url (string)
    returns:    unique player ID (string)
    '''
    #every player ID used by the NHL website and API is 7 digits long
    split = url.rsplit("-")
    return split[-1]
def placeToStats(place_list):
    '''
    purpose:    takes a list of players from the webscraped csv and returns a list of the stats of players in the list
    parameters: place_list (a pandas dataframe of tuples)
    returns:    list of player IDs
    '''
    ids = []
    seasons = place_list.iloc[:,0]
    players = place_list.iloc[:,1]
    seasons = seasons.tolist()
    players = players.tolist()

    for i in range(len(players)):
        playerTuple = ast.literal_eval(players[i])    
        if type(playerTuple) != tuple:      #working with an entry where multiple players won this award
            for entry in playerTuple:
                id = extractPlayerID(entry[0])
                ids.append((seasons[i],id))
            continue
        
        id = extractPlayerID(playerTuple[0])
        ids.append((seasons[i],id))
    return ids

#### Milestone 1: Get relevant data of relevant skaters from multiple seasons, their summary and EDGE statistics - will be exported to data/api folder for other events

In [25]:
#1.1 get all skater data from one season (2023-2024). #help(client.stats.skater_stats_summary)
def fetchSkaterStats(year, csv=False):
    '''
    purpose:        fetches all summary skater stats of a given year and compresses into the desired format
    parameters:     year (string), csv (boolean)
    returns:        a dataframe OR csv of all player stats of that given year
    '''
    year_df = []

    #format year for the filter (is a string when inputted)
    if len(year) == 4:
        int_year = int(year)
        interval = (int_year,int_year+1)
        year_interval = str(interval[0]) + str(interval[1])
    elif len(year) == 8:    #20252026
        year_interval = year
    else:
        raise SyntaxError("requires yyyy or yyyyyyyy format of season year")
    
    #since there are ~900 skaters each season, must account for pagination
    #earlier testing indicates a 100 entry limit per request
    for i in range(10):
        startMark = 100*i
        endMark = 100*(i+1)
        statChunk = client.stats.skater_stats_summary(
            start_season=year_interval,
            end_season=year_interval,
            start = startMark,
            limit = endMark
        )
        for record in statChunk:
            year_df.append(record)

    df = pd.DataFrame(year_df)

    if csv == True:
        df.to_csv(f'../data/api/skaters/skaters{year_interval}.csv',index=False)
    else:
        return df
    

In [26]:
#place relevant season files into the data/api/skaters folder -- DON'T RUN THIS ANYMORE
'''
for season in rocket_richard['szn']:
    first_year = season.split("-")
    first_year = first_year[0]
    fetchSkaterStats(first_year, csv=True)
'''

'\nfor season in rocket_richard[\'szn\']:\n    first_year = season.split("-")\n    first_year = first_year[0]\n    fetchSkaterStats(first_year, csv=True)\n'

In [27]:
#this is a separate code block that supplements all the missing skater seasons csvs in data/api/skates - written 07/28/26 after the pipeline was completed for singular winner prediction
#DO NOT RUN THIS ANYMORE
'''
missingSeasons = ['2003-2004','2009-2010','1998-1999','2001-2002']
for missing in missingSeasons:
    start_year = missing.split("-")
    start_year = start_year[0]
    fetchSkaterStats(start_year,csv=True)
'''

'\nmissingSeasons = [\'2003-2004\',\'2009-2010\',\'1998-1999\',\'2001-2002\']\nfor missing in missingSeasons:\n    start_year = missing.split("-")\n    start_year = start_year[0]\n    fetchSkaterStats(start_year,csv=True)\n'

### 1.2 combine available columns of a player with their given edge stats as well for features
#### note: adding EDGE statistics will increase resources for training and performance, so for now I will keep edge stats for players seperate and possibly do two methods of prediction: one based off general skater statistics, on another based on general skater statistics + EDGE stats; it'd be a good takeaway of findings

In [28]:
#1.3 combine summary + edge stats for all seasons listen in rocketrichard.csv
rocket_richard['szn']

1     2025-26
3     2024-25
5     2023-24
7     2022-23
9     2021-22
11    2020-21
13    2019-20
15    2018-19
17    2017-18
19    2016-17
21    2015-16
23    2014-15
25    2013-14
27    2012-13
29    2011-12
31    2010-11
35    2008-09
37    2007-08
39    2006-07
41    2005-06
45    2002-03
49    2000-01
51    1999-00
Name: szn, dtype: object

#### Milestone 2: Build the targets (label supervised data), identify winners, create features, build the dataset for training / testing splits

In [29]:
def extractPlayerID(url):

    '''
    purpose:    parses a url and returns that player's id (used by the NHL API) 
    parameters: player url (string)
    returns:    unique player ID (string)
    '''
    #every player ID used by the NHL website and API is 7 digits long
    split = url.rsplit("-")
    return split[-1]

def placeToStats(place_list):
    '''
    purpose:    takes a list of players from the webscraped csv and returns a list of the stats of players in the list
    parameters: place_list (a pandas dataframe of tuples)
    returns:    list of player IDs
    '''
    ids = []
    seasons = place_list.iloc[:,0]
    players = place_list.iloc[:,1]
    seasons = seasons.tolist()
    players = players.tolist()

    for i in range(len(players)):
        playerTuple = ast.literal_eval(players[i])    
        if type(playerTuple) != tuple:      #working with an entry where multiple players won this award
            for entry in playerTuple:
                id = extractPlayerID(entry[0])
                ids.append((seasons[i],id))
            continue
        
        id = extractPlayerID(playerTuple[0])
        ids.append((seasons[i],id))
    return ids

first_place = rocket_richard[['szn','winner']]
first_ids = placeToStats(first_place)
#first_ids

second_place = rocket_richard[['szn', 'runner_up']]
second_ids = placeToStats(second_place)

third_place = rocket_richard[['szn', 'finalist']]
third_ids = placeToStats(third_place)

#print(len(first_ids),len(second_ids),len(third_ids))    #24 / 23 / 25
#max(len(first_ids),len(second_ids), len(third_ids))
#second_ids
#third_ids

#for i in range(len(first_ids)):
#    print(first_ids[i], second_ids[i], third_ids[i])

In [30]:
def labelWinners(year, rank=False):
    '''
    purpose:    fetches a dataset of skaters and adds two columns: average TOI (extra feature) and either rrWinner or rrRank (target features) 
    parameters: year (string) of a valid RR winner year; in yyyy or yyyyyyyy format and rank (boolean) if the dev wants labels to be one-hot encoded OR ranked by top 3 finalists
    returns:    returns a dataframe of the selected year skaters with the labeled RR winner/finalists
    '''
    #format year for the filter (is a string when inputted)
    if len(year) == 4:
        int_year = int(year)
        interval = (int_year,int_year+1)
        year_interval = str(interval[0]) + str(interval[1])
    elif len(year) == 8:    #20252026
        year_interval = year
    else:
        raise SyntaxError("requires yyyy or yyyyyyyy format of season year")
    
    csv_path = f"../data/api/skaters/skaters{year_interval}.csv"
    df = pd.read_csv(csv_path)

    #add averageTOI
    df['averageTOI'] = np.zeros(df.shape[0])
    df['averageTOI'] = (df['timeOnIcePerGame']/60)
    startYear = year_interval[:4]               #get the first year in this season's year interval

    #add rr Winner ONLY
    if rank == False:       
        df['rrWinner'] = np.zeros(df.shape[0])      #create a new column for ranking
        for entry in first_ids:
            split = entry[0].rsplit("-")
            if str(split[0]) == (startYear):
                winner = entry[1]
                break
        df.loc[df['playerId'] == int(winner), 'rrWinner'] = 1   #modify the entry directly

    #add rr finalists
    else:
        df['rrRank'] = np.zeros(df.shape[0])      #create a new column for ranking
        for first in first_ids:
            season_year = first[0]
            season_year = season_year.rsplit("-")
            season_year = season_year[0]
            if season_year == startYear:
                winner = first[1]
                df.loc[df['playerId'] == int(winner), 'rrRank'] = 1
        
        for second in second_ids:
            season_year = second[0]
            season_year = season_year.rsplit("-")
            season_year = season_year[0]
            if season_year == startYear:
                runner_up = second[1]
                df.loc[df['playerId'] == int(runner_up), 'rrRank'] = 2
            
        for third in third_ids:
            season_year = third[0]
            season_year = season_year.rsplit("-")
            season_year = season_year[0]
            if season_year == startYear:
                finalist = third[1]
                df.loc[df['playerId'] == int(finalist), 'rrRank'] = 3
    return df

ska23 = labelWinners("2023", False)

#ska23.loc[ska23['rrRank'] != 0]     #shows Matthews, Reinhart and HYMAN; WORKS



I notice that a lot of entries for players with NaN values are players that are frequently sent down to the AHL and called up to the NHL, so its best to just omit them since they have little to no chance of actually winning the award as compared to players playing their full season in the NHL

In [31]:
#filter out columns that may influence training when it shouldn't
to_drop = ['lastName','positionCode','seasonId','skaterFullName','teamAbbrevs', 'shGoals', 'shPoints']   #identifying features like lastName and 

feature_set = ska23.drop(columns=to_drop)                                    #drop non-relevant columns
feature_set = feature_set.dropna()                                           #drop player entries with NaN values
feature_set.loc[feature_set['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
feature_set.loc[feature_set['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model

feature_set

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,...,points,pointsPerGame,ppGoals,ppPoints,shootingPct,shootsCatches,shots,timeOnIcePerGame,averageTOI,rrWinner
0,100,31,91,0.50000,6,81,44,1,22,8476453,...,144,1.77777,13,53,0.14379,0,306,1300.1234,21.668723,0.0
1,89,41,92,0.46194,9,82,51,2,42,8477492,...,140,1.70731,10,48,0.12592,1,405,1368.8536,22.814227,0.0
2,100,24,87,0.51098,5,76,32,1,30,8478402,...,132,1.73684,7,44,0.12167,0,263,1281.5526,21.359210,0.0
3,71,38,75,0.10000,5,82,49,1,24,8478550,...,120,1.46341,11,44,0.16171,1,303,1207.1341,20.118902,0.0
4,63,35,75,0.33333,5,82,47,0,47,8477956,...,110,1.34146,12,35,0.12303,1,382,1195.7804,19.929673,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
918,0,0,0,0.00000,0,8,0,0,6,8483395,...,0,0.00000,0,0,0.00000,0,4,708.3750,11.806250,0.0
919,0,0,0,0.50000,0,9,0,0,2,8479514,...,0,0.00000,0,0,0.00000,0,13,935.2222,15.587037,0.0
920,0,0,0,0.00000,0,9,0,0,0,8482751,...,0,0.00000,0,0,0.00000,1,10,560.3333,9.338888,0.0
921,0,0,0,0.50000,0,9,0,0,6,8482784,...,0,0.00000,0,0,0.00000,0,10,633.2222,10.553703,0.0


In [32]:
#splice training and test sets

trainingSets = []
testingSets = []
#to_drop = ['lastName','playerId','positionCode','seasonId','skaterFullName','teamAbbrevs', 'shGoals', 'shPoints']
to_drop = ['positionCode','skaterFullName','teamAbbrevs', 'shGoals', 'shPoints']

for year in rocket_richard['szn']:
    if year == "2025-26" or year == "2024-25" or year == "2023-24":     #save the last 3 years for testing predictions
        split = year.rsplit("-")
        if ranks == True:
            modifiedDf = labelWinners(year=split[0], rank=True)
        else:
            modifiedDf = labelWinners(year=split[0], rank=False)
        feature_df = modifiedDf.drop(columns=to_drop)
        feature_df = feature_df.dropna()
        feature_df .loc[feature_df ['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
        feature_df .loc[feature_df ['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model
        testingSets.append(feature_df )
    else:
        #print(year)
        split = year.rsplit("-")
        if ranks == True:
            modifiedDf = labelWinners(year=split[0], rank=True)
        else:
            modifiedDf = labelWinners(year=split[0], rank=False)
        feature_df = modifiedDf.drop(columns=to_drop)
        feature_df = feature_df.dropna()
        feature_df.loc[feature_df['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
        feature_df.loc[feature_df['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model
        trainingSets.append(feature_df)

masterTraining = pd.DataFrame()
for train in trainingSets:
    masterTraining = pd.concat([masterTraining,train])

#print(masterTraining.shape)    #(11150,20)
#print(masterTraining.loc[masterTraining['rrWinner']==1])

masterTesting = pd.DataFrame()
for test in testingSets:
    masterTesting = pd.concat([masterTesting, test])
#print(masterTesting.shape)      #(1664,20)

masterTesting

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,lastName,otGoals,penaltyMinutes,...,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,timeOnIcePerGame,averageTOI,rrRank
0,90,34,82,0.49512,4,82,48,McDavid,1,44,...,1.68292,13,54,20252026,0.15686,0,306,1379.1219,22.985365,3.0
1,86,35,92,0.00000,8,76,44,Kucherov,2,50,...,1.71052,8,37,20252026,0.19047,0,231,1220.1052,20.335087,0.0
2,74,42,97,0.51049,7,80,53,MacKinnon,1,39,...,1.58750,11,30,20252026,0.15142,1,350,1335.7125,22.261875,1.0
3,70,37,82,0.48638,5,82,45,Celebrini,2,44,...,1.40243,8,33,20252026,0.15679,0,287,1278.9512,21.315853,0.0
4,67,29,80,0.46661,6,82,36,Scheifele,1,43,...,1.25609,7,22,20252026,0.20571,1,175,1290.2439,21.504065,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
918,0,0,0,0.00000,0,8,0,Bains,0,6,...,0.00000,0,0,20232024,0.00000,0,4,708.3750,11.806250,0.0
919,0,0,0,0.50000,0,9,0,Pitlick,0,2,...,0.00000,0,0,20232024,0.00000,0,13,935.2222,15.587037,0.0
920,0,0,0,0.00000,0,9,0,Winterton,0,0,...,0.00000,0,0,20232024,0.00000,1,10,560.3333,9.338888,0.0
921,0,0,0,0.50000,0,9,0,Dean,0,6,...,0.00000,0,0,20232024,0.00000,0,10,633.2222,10.553703,0.0


#### Milestone 3: Train the first model for the first award - uses sklearn Pipeline

In [33]:
#logistic regression first
from sklearn.linear_model import LogisticRegression
if ranks == False:
    X = masterTraining.drop(columns=['rrWinner','playerId']) #everything but the target feature
    Y = masterTraining['rrWinner']              #target vector
else:
    X = masterTraining.drop(columns=['rrRank','playerId'])   #everything but target feature
    Y = masterTraining['rrRank']

print(X.shape)


(11150, 21)


In [34]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

if ranks == False:
        train_x = masterTraining.drop(columns=['lastName','rrWinner','playerId','seasonId']) #everything but the target feature
        train_y = masterTraining['rrWinner']              #target vector
else:
        train_x = masterTraining.drop(columns=['lastName','rrRank','playerId','seasonId'])
        train_y = masterTraining['rrRank'] 

print(train_x.shape)
model = LogisticRegression(max_iter=15000, class_weight="balanced")
model.fit(train_x, train_y)
train_x

(11150, 19)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 13367 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,plusMinus,points,pointsPerGame,ppGoals,ppPoints,shootingPct,shootsCatches,shots,timeOnIcePerGame,averageTOI
0,89,39,75,0.51928,11,82,64,2,36,22,153,1.86585,21,71,0.18181,0,352,1343.1341,22.385568
1,76,19,64,0.54897,11,80,52,1,24,7,128,1.60000,32,62,0.21052,0,247,1304.0375,21.733958
2,83,22,63,1.00000,4,82,30,0,36,-2,113,1.37804,8,50,0.11070,0,271,1207.9390,20.132317
3,52,43,76,0.42105,13,82,61,4,38,34,113,1.37804,18,37,0.14987,1,407,1173.8658,19.564430
4,69,30,77,0.44434,9,71,42,2,30,29,111,1.56338,12,34,0.11475,1,366,1339.0422,22.317370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824,0,0,0,0.55769,0,9,0,0,5,-2,0,0.00000,0,0,0.00000,0,2,459.1111,7.651852
825,0,0,0,0.58064,0,10,0,0,6,-2,0,0.00000,0,0,0.00000,0,3,446.9000,7.448333
830,0,0,0,0.00000,0,12,0,0,4,-6,0,0.00000,0,0,0.00000,0,12,552.7500,9.212500
831,0,0,0,0.45161,0,13,0,0,14,-5,0,0.00000,0,0,0.00000,0,5,503.1538,8.385897


#### Milestone 4: Test against testSets (for seasons 2023-2026)

In [35]:
#do 3 seperate tests: 1 for 2023-2024, 1 for 2024-2025, 1 for 2025-2026
#2023-2024
test2023 = testingSets[2]
if ranks == False:
    test_x = test2023.drop(columns=['lastName','rrWinner','playerId','seasonId'])
    test_y = test2023['rrWinner']
    print("only one winner")
else:
    test_x = test2023.drop(columns=['lastName','rrRank','playerId','seasonId'])
    test_y = test2023['rrRank']
    print("top3")

pred_y = model.predict(test_x)
predictions = pd.Series(pred_y)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = test2023.join(predictions)


top3


In [36]:
#2024-2025 test
'''
test2024 = testingSets[1]
if ranks == False:
    test_x = test2024.drop(columns=['lastName','rrWinner','playerId','seasonId'])
    test_y = test2024['rrWinner']
else:
    test_x = test2024.drop(columns=['lastName','rrRank','playerId','seasonId'])
    test_y = test2024['rrRank']

pred_y = model.predict(test_x)
predictions = pd.Series(pred_y)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = test2024.join(predictions)
'''


"\ntest2024 = testingSets[1]\nif ranks == False:\n    test_x = test2024.drop(columns=['lastName','rrWinner','playerId','seasonId'])\n    test_y = test2024['rrWinner']\nelse:\n    test_x = test2024.drop(columns=['lastName','rrRank','playerId','seasonId'])\n    test_y = test2024['rrRank']\n\npred_y = model.predict(test_x)\npredictions = pd.Series(pred_y)\npredictions = predictions.rename('predictions')\npredictions = predictions.to_frame()\nresults = test2024.join(predictions)\n"

In [37]:
#2025-2026 test
'''
test2025 = testingSets[0]
if ranks == False:
    test_x = test2025.drop(columns=['lastName','rrWinner','playerId','seasonId'])
    test_y = test2025['rrWinner']
else:
    test_x = test2025.drop(columns=['lastName','rrRank','playerId','seasonId'])
    test_y = test2025['rrRank']

pred_y = model.predict(test_x)
predictions = pd.Series(pred_y)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = test2025.join(predictions)
'''

"\ntest2025 = testingSets[0]\nif ranks == False:\n    test_x = test2025.drop(columns=['lastName','rrWinner','playerId','seasonId'])\n    test_y = test2025['rrWinner']\nelse:\n    test_x = test2025.drop(columns=['lastName','rrRank','playerId','seasonId'])\n    test_y = test2025['rrRank']\n\npred_y = model.predict(test_x)\npredictions = pd.Series(pred_y)\npredictions = predictions.rename('predictions')\npredictions = predictions.to_frame()\nresults = test2025.join(predictions)\n"

In [38]:
#view influences on the model
feature_names = train_x.columns
feature_names
coefficients = pd.Series(model.coef_[0],index=feature_names)
print(coefficients.sort_values())

goals              -0.676900
ppGoals            -0.523031
points             -0.466363
evGoals            -0.275288
shootsCatches      -0.048342
faceoffWinPct      -0.024630
averageTOI          0.000123
timeOnIcePerGame    0.007399
shots               0.011797
plusMinus           0.014965
penaltyMinutes      0.029403
pointsPerGame       0.032089
shootingPct         0.037080
gameWinningGoals    0.129441
evPoints            0.164909
assists             0.210537
otGoals             0.271509
ppPoints            0.377948
gamesPlayed         0.467298
dtype: float64


In [39]:
#train_x.corr()["goals"].sort_values()
#train_x.corrwith(train_y).sort_values()

In [40]:
if ranks == False:
    show = results.loc[results['predictions'] == 1]
else:
    print("ben")
    show = results.loc[results['predictions'] != 0]
    show = show.dropna()
    hyman = results.loc[results['rrRank'] != 0]


with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
    display(show)  

ben


,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,lastName,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,timeOnIcePerGame,averageTOI,rrRank,predictions
0,100,31,91,0.50000,6,81,44,Kucherov,1,22,8476453,8,144,1.77777,13,53,20232024,0.14379,0,306,1300.1234,21.668723,0.0,2.0
1,89,41,92,0.46194,9,82,51,MacKinnon,2,42,8477492,35,140,1.70731,10,48,20232024,0.12592,1,405,1368.8536,22.814227,0.0,2.0
3,71,38,75,0.10000,5,82,49,Panarin,1,24,8478550,18,120,1.46341,11,44,20232024,0.16171,1,303,1207.1341,20.118902,0.0,2.0
4,63,35,75,0.33333,5,82,47,Pastrnak,0,47,8477956,21,110,1.34146,12,35,20232024,0.12303,1,382,1195.7804,19.929673,0.0,2.0
5,38,51,77,0.53449,8,81,69,Matthews,3,20,8479318,31,107,1.32098,18,29,20232024,0.18699,0,369,1257.9876,20.966460,1.0,1.0
6,65,20,65,0.56926,7,81,41,Draisaitl,2,76,8477934,26,106,1.30864,21,39,20232024,0.18894,0,217,1241.7283,20.695472,0.0,3.0
7,62,28,64,0.53816,9,80,42,Rantanen,0,50,8478420,19,104,1.30000,14,40,20232024,0.15498,0,271,1374.3375,22.905625,0.0,3.0
10,50,27,55,0.00000,8,75,46,Kaprizov,2,36,8478864,11,96,1.28000,19,41,20232024,0.16606,0,277,1294.6133,21.576888,0.0,3.0
11,52,32,71,0.58226,3,82,42,Crosby,0,40,8471675,7,94,1.14634,10,23,20232024,0.15107,0,278,1205.2073,20.086788,0.0,2.0
12,46,35,62,0.50000,11,82,48,Forsberg,3,43,8476887,16,94,1.14634,13,32,20232024,0.13832,1,347,1134.3292,18.905487,0.0,2.0


In [41]:
#now, pick the top 1 or 3 players in terms of goals, and that is the prediction
#2023
if ranks == False:      #get top 1
    #print(show.goals.index)     #these are the indexes of the predicted players
    indexes = show.goals.sort_values(ascending=False).index
    print(indexes)
    predicted_winners_descending = test2023.loc[indexes]
else:                   #get top 3
    #indexes = show.goals.sort_values(ascending=False).index
    #print(indexes)
    #predicted_winners_descending = test2023.loc[indexes]
    #predicted_winners_descending = predicted_winners_descending.loc[predicted_winners_descending['predictions'] != 0]    #predicted Auston Matthews and Sam Reinhart only (these are correct), where's Zach Hyman? 
    show = show.dropna()

#show.loc[show['predictions'] != 0]      #to be uncommented when ranks = True
predicted_winners_descending             #to be uncommented when ranks = False

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,lastName,otGoals,penaltyMinutes,...,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,timeOnIcePerGame,averageTOI,rrWinner
5,38,51,77,0.53449,8,81,69,Matthews,3,20,...,1.32098,18,29,20232024,0.18699,0,369,1257.9876,20.966460,1.0
13,37,25,55,0.44150,11,82,57,Reinhart,3,31,...,1.14634,27,34,20232024,0.24463,1,233,1217.9878,20.299797,0.0
4,63,35,75,0.33333,5,82,47,Pastrnak,0,47,...,1.34146,12,35,20232024,0.12303,1,382,1195.7804,19.929673,0.0
10,50,27,55,0.00000,8,75,46,Kaprizov,2,36,...,1.28000,19,41,20232024,0.16606,0,277,1294.6133,21.576888,0.0
25,41,21,42,0.56043,6,79,40,Stamkos,0,34,...,1.02531,19,39,20232024,0.15267,1,262,1093.7088,18.228480,0.0


In [42]:
if ranks == True:
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(ska23.loc[ska23['rrRank'] != 0])

KeyError: 'rrRank'

#### Milestone 5: Predict on midseason (moreso a later thing to do..?)